## Stats on medication dataset

In [ ]:
import pandas as pd
import numpy as np
import random

## Part 1: Diccionari de fàrmacs. Codis ATC (7 caràcters)

In [ ]:
# Mapegem cada grup de diagnòstic a una llista de medicaments reals
diccionari_farmacs = {
    "Trastorns de l'ansietat": [
        {'atc': 'N05BA01', 'nom': 'Diazepam', 'preu_base': 2.50},
        {'atc': 'N05BA06', 'nom': 'Lorazepam', 'preu_base': 3.10},
        {'atc': 'N05BA12', 'nom': 'Alprazolam', 'preu_base': 4.20}
    ],
    "Trastorns de l'estat d'ànim": [
        {'atc': 'N06AB10', 'nom': 'Escitalopram', 'preu_base': 15.10}, 
        {'atc': 'N06AB06', 'nom': 'Sertralina', 'preu_base': 18.50},
        {'atc': 'N06AB03', 'nom': 'Fluoxetina', 'preu_base': 12.30}
    ],
    "Esquizofrènia i trastorns psicòtics": [
        {'atc': 'N05AD01', 'nom': 'Haloperidol', 'preu_base': 8.50},
        {'atc': 'N05AH02', 'nom': 'Clozapina', 'preu_base': 42.10},
        {'atc': 'N05AH03', 'nom': 'Olanzapina', 'preu_base': 35.20},
        {'atc': 'N05AH04', 'nom': 'Quetiapina', 'preu_base': 32.40},
        {'atc': 'N05AX08', 'nom': 'Risperidona', 'preu_base': 28.50},
        {'atc': 'N05AX12', 'nom': 'Aripiprazol', 'preu_base': 55.00},
        {'atc': 'N05AX13', 'nom': 'Paliperidona', 'preu_base': 65.20},
        {'atc': 'N05AE05', 'nom': 'Lurasidona', 'preu_base': 58.00}
    ],
    "Trastorns per consum de substàncies": [
        {'atc': 'N07BB01', 'nom': 'Disulfiram', 'preu_base': 22.30},
        {'atc': 'N07BA01', 'nom': 'Nicotina', 'preu_base': 30.50},
        {'atc': 'N05BA06', 'nom': 'Lorazepam', 'preu_base': 3.10}
    ]
}

## Part 2: Càrrega de dades i selecció de la mostra

In [ ]:
path_diag = r'C:\Users\LRAJAG\Desktop\Data_sintetica\csv\5_Psychiatric_Diagnosis\diagnostics_psiquiatrics_detallat_100k.csv'
df_diag = pd.read_csv(path_diag)

# Seleccionem 1.000 pacients aleatoris que TINGUIN un diagnòstic
# Filtrem aquells que no siguin "Sense diagnòstic" o "Sense especificació"
df_casos = df_diag[~df_diag['grup_diagnostic'].isin(["Sense diagnòstic", "Sense especificació"])].sample(1000, random_state=42)

## Part 3: Generació de dispensacions

In [ ]:
mesos = [f"{a}{m:02d}" for a in range(2010, 2017) for m in range(1, 13)]
dades_dispensacio = []

print(f"Generant dispensacions per a 1.000 pacients...")

for _, pacient in df_casos.iterrows():
    id_p = pacient['id_pacient']
    grup = pacient['grup_diagnostic']
    
    # Si el grup té medicaments assignats al nostre diccionari
    if grup in diccionari_farmacs:
        # Assignem UN medicament fix per a tota la durada de l'estudi (tractament crònic)
        farmac = random.choice(diccionari_farmacs[grup])
        
        for mes in mesos:
            # Simulem un 85% d'adherència (no tots els mesos van a la farmàcia)
            if random.random() < 0.85:
                # Variació petita de preu (inflació/canvis de mercat simulats)
                pvp = round(farmac['preu_base'] * random.uniform(0.98, 1.05), 2)
                # Aportació típica (10% per a molts tractaments crònics)
                aportacio = round(pvp * 0.10, 2)
                
                dades_dispensacio.append({
                    'id_pacient': id_p,
                    'mes_dispensacio': mes,
                    'codi_atc': farmac['atc'],
                    'nom_farmac': farmac['nom'],
                    'num_envasos': 1,
                    'dies_tractament': 30,
                    'preu_pvp': pvp,
                    'aportacio_usuari': aportacio
                })

## Part 4: Creació del fitxer i estadístiques

In [ ]:
df_farmacia = pd.DataFrame(dades_dispensacio)

# Guardar el CSV
path_sortida = r'C:\Users\LRAJAG\Desktop\Data_sintetica\csv\6_Pharmacy\dispensacions_farmaceutiques_1k.csv'
df_farmacia.to_csv(path_sortida, index=False)

print(f"\n✅ Procés finalitzat!")
print(f"Total de registres generats: {len(df_farmacia):,}")
print(f"Mida estimada del fitxer: {round(len(df_farmacia)*80/1024/1024, 2)} MB (Apte per a GitHub)")
print("\nMostra de les primeres 5 files:")
print(df_farmacia.head())